# Executive Summary

**Objective:** 
To integrate geographical metadata, resolve string formatting inconsistencies, and engineer normalized metrics, transforming the raw data into a finalized state ready for exploratory data analysis.

**Data Flow:**
*   **Inputs:** 
    * `data\raw\internship_positions.parquet`
    * `data\external\administrative_divisions.parquet`
*   **Output:** `data\interim\internship_postings.parquet` (Exported to the interim directory to preserve datatypes)

**Key Operations Performed:**
1. **Data Integration (Province Mapping) [<u>[click]</u>](#1-data-integration):** Mapped raw job locations to their respective provinces by joining against the administrative divisions dictionary.
    * **String Cleaning [<u>[click]</u>](#11-string-cleaning):** Resolved mismatching geographic keys by standardizing text (lowercasing, stripping punctuation, and removing extraneous words).
    * **Manual Overrides [<u>[click]</u>](#12-manual-overrides):** Applied a manual dictionary mapping to achieve 100% province mapping with zero null values.
2. **Data Conversion [<u>[click]</u>](#2-data-conversion):** Cast `weekly_working_day` to a categorical datatype due to its low cardinality (2 unique values).
3. **Feature Engineering [<u>[click]</u>](#3-feature-engineering):**
    * Engineered the `acceptance_percentage` column (`100 * approved_quota / (1 + applicant_count)`).
    * Categorized `job_title` into a new `job_category` feature to reduce cardinality.
    * Engineered 10 new boolean flag columns (e.g., `allows_it_and_computer_majors`) by grouping over 1,300 distinct majors in the `allowed_major` column using regex pattern matching.
    * Binned all heavy right-skewed numericals (`requested_quota`, `approved_quota`, `applicant_count`, `acceptance_percentage`) to capture "whale" postings in an "Extreme" category without deleting them.
    * One-hot encoded `education_level` for downstream stakeholder consumption.
    * Extracted a new binary flag, `allows_all_majors`, by parsing the `job_description` column.
4. **Schema Finalization [<u>[click]</u>](#4-schema-finalization):** Reorganized the final 14 columns into a logical analytical structure before exporting.

# Setup & Imports

In [107]:
# Import libraries
import numpy as np
import pandas as pd

from src.config import EXTERNAL_DATA_DIR, INTERIM_DATA_DIR, RAW_DATA_DIR

In [108]:
# Load datasets
adm_divisions = pd.read_parquet(EXTERNAL_DATA_DIR / "administrative_divisions.parquet")
internship_positions = pd.read_parquet(RAW_DATA_DIR / "internship_positions.parquet")

# 1. Data Integration
Mapping raw job locations to their respective provinces by joining against the administrative divisions dictionary.

In [109]:
# Convert a regency-province table into a dictionary
adm_divisions_dict = dict(zip(adm_divisions["regency"], adm_divisions["province"]))


In [110]:
# Create a new column, `province`, by mapping with a dictionary
internship_positions["province"] = internship_positions["job_location"].map(adm_divisions_dict)
display(internship_positions.head())

,job_id,published_at,job_title,company,job_location,education_level,allowed_major,job_description,weekly_working_day,requested_quota,approved_quota,applicant_count,province
0,a240f2ba-12c0-4958-b416-c3e9c1d4e344,2026-07-16T12:58:55+07:00,PSIKOLOG,RUMAH TAHANAN NEGARA KELAS IIB SIBUHUAN,Kab. Padang Lawas,Profession,Psikologi,1. Melakukan asesmen psikologis terhadap anak ...,6,1,1,0,Sumatera Utara
1,a240f336-b6b7-47fe-8096-5b4c2eb2ed1f,2026-07-16T12:58:55+07:00,PSIKIATER,RUMAH TAHANAN NEGARA KELAS IIB SIBUHUAN,Kab. Padang Lawas,Profession,Kedokteran,1. Menangani gangguan kesehatan jiwa warga bin...,6,1,1,0,Sumatera Utara
2,a242e6a4-2c7a-4ce4-8a57-7aff601a5e2c,2026-07-16T12:57:40+07:00,PSIKIATER,RUMAH TAHANAN NEGARA KELAS IIB SALATIGA,Kota Salatiga,Profession,Kedokteran,1. Menangani gangguan kesehatan jiwa warga bin...,5,1,1,0,Jawa Tengah
3,a24138aa-bcf6-4940-bfaf-dfe5dbf66ca6,2026-07-16T12:50:42+07:00,PERAWAT KESEHATAN,LEMBAGA PEMASYARAKATAN KELAS III SUKAMARA,Kab. Sukamara,Bachelor,Ilmu Gizi,1. Memberikan perawatan kesehatan umum dan tin...,6,1,1,0,Kalimantan Tengah
4,a23f7c52-9123-46e9-bb5e-be376b1d77f2,2026-07-16T12:38:52+07:00,Psikiater,LEMBAGA PEMASYARAKATAN KELAS III ARJASA,Kab. Sumenep,Bachelor,Kedokteran,1. Menangani gangguan kesehatan jiwa warga bin...,6,1,1,0,Jawa Timur


In [111]:
# Find regencies from `internship_positions` that have no match 
# with their corresponding provinces
display(
    internship_positions[
        internship_positions["province"].isnull()
    ].groupby("job_location")["job_id"].count().sort_values(ascending=False)
)

job_location
Kota Batam                          221
Kota Palangkaraya                   120
Kota Dumai                           77
Kab. Pangkajene Kepulauan            57
Kota Bau Bau                         56
Kota Banjarbaru                      48
Kab. Siak                            46
Kota Lubuk Linggau                   43
Kab. Gunungkidul                     42
Kab. Karangasem                      39
Kota Sawahlunto                      38
Kab. Batanghari                      31
Kab. Tulang Bawang                   28
Kota Pematangsiantar                 25
Unknown Location                     24
Kab. Kotabaru                        24
Kab. Banyuasin                       23
Kab. Labuhanbatu                     23
Kota Pare Pare                       20
Kab. Tojo Una Una                    18
Kab. Pahuwato                        17
Kab. Kep. Siau Tagulandang Biaro     17
Kepulauan Tanimbar                   15
Kab Timor Tengah Selatan             15
Kab. Toli Toli             

In [112]:
# Get all regencies from `adm_divisions` that have no match
# with job locations from `internship_positions`
matched_regencies = list(set(internship_positions[
    ~internship_positions.province.isnull()
]["job_location"].to_list()))

unmatched_regencies = list(set(adm_divisions.regency.to_list()) - set(matched_regencies))

display(
    adm_divisions[
        adm_divisions.regency.isin(unmatched_regencies)
    ][["regency", "province"]].sort_values("regency", ascending=False)
)

,regency,province
70,Kota Sawah Lunto,Sumatera Barat
50,Kota Pematang Siantar,Sumatera Utara
420,Kota Parepare,Sulawesi Selatan
341,Kota Palangka Raya,Kalimantan Tengah
114,Kota Lubuklinggau,Sumatera Selatan
...,...,...
90,Kab. Batang Hari,Jambi
104,Kab. Banyu Asin,Sumatera Selatan
144,Kab. Bangka Selatan,Kepulauan Bangka Belitung
385,Kab. Banggai Kepulauan,Sulawesi Tengah


## 1.1 String Cleaning

In [113]:
# Fallback mapping: Stripping punctuation and whitespace resolves mismatches
# caused by inconsistent data entry on the scraped platform.
plain_adm_div_dict = dict(
    zip(
        adm_divisions["regency"]
        .str.lower()
        .str.replace(r"\sdan\s", " ", case=False, regex=True)
        .str.replace(r"\W", "", regex=True),
        adm_divisions["province"],
    )
)

display(plain_adm_div_dict)

mask = internship_positions["province"].isnull()
internship_positions.loc[mask, "province"] = (
    internship_positions.loc[mask, "job_location"]
    .str.lower()
    .str.replace(r"\sdan\s", " ", case=False, regex=True)
    .str.replace(r"\W", "", regex=True)
    .map(plain_adm_div_dict)
)

{'kabsimeulue': 'Aceh',
 'kabacehsingkil': 'Aceh',
 'kabacehselatan': 'Aceh',
 'kabacehtenggara': 'Aceh',
 'kabacehtimur': 'Aceh',
 'kabacehtengah': 'Aceh',
 'kabacehbarat': 'Aceh',
 'kabacehbesar': 'Aceh',
 'kabpidie': 'Aceh',
 'kabbireuen': 'Aceh',
 'kabacehutara': 'Aceh',
 'kabacehbaratdaya': 'Aceh',
 'kabgayolues': 'Aceh',
 'kabacehtamiang': 'Aceh',
 'kabnaganraya': 'Aceh',
 'kabacehjaya': 'Aceh',
 'kabbenermeriah': 'Aceh',
 'kabpidiejaya': 'Aceh',
 'kotabandaaceh': 'Aceh',
 'kotasabang': 'Aceh',
 'kotalangsa': 'Aceh',
 'kotalhokseumawe': 'Aceh',
 'kotasubulussalam': 'Aceh',
 'kabnias': 'Sumatera Utara',
 'kabmandailingnatal': 'Sumatera Utara',
 'kabtapanuliselatan': 'Sumatera Utara',
 'kabtapanulitengah': 'Sumatera Utara',
 'kabtapanuliutara': 'Sumatera Utara',
 'kabtobasamosir': 'Sumatera Utara',
 'kablabuhanbatu': 'Sumatera Utara',
 'kabasahan': 'Sumatera Utara',
 'kabsimalungun': 'Sumatera Utara',
 'kabdairi': 'Sumatera Utara',
 'kabkaro': 'Sumatera Utara',
 'kabdeliserdang': '

In [114]:
# Check for the missing values again by finding regencies from `internship_positions`
# that have no match with their corresponding provinces
display(
    internship_positions[
        internship_positions["province"].isnull()
    ].groupby("job_location")["job_id"].count().sort_values(ascending=False)
)

job_location
Unknown Location                    24
Kab. Kep. Siau Tagulandang Biaro    17
Kab. Pahuwato                       17
Kepulauan Tanimbar                  15
Kab. Mahakam Ulu                     1
Name: job_id, dtype: int64

## 1.2 Manual Overrides

In [115]:
# Manual overrides for edge cases missing from the standard division dataset
manual_dict = {
    "Kab. Kep. Siau Tagulandang Biaro": "Sulawesi Utara",
    "Kab. Mahakam Ulu": "Kalimantan Timur",
    "Kab. Pahuwato": "Gorontalo",
    "Kepulauan Tanimbar": "Maluku",
    "Unknown Location": "Unknown Location",
}

mask = internship_positions["province"].isnull()
internship_positions.loc[mask, "province"] = internship_positions.loc[
    mask, "job_location"
].map(manual_dict)

In [116]:
# Check for the missing values again by finding regencies from `internship_positions`
# that have no match with their corresponding provinces
display(
    internship_positions[
        internship_positions["province"].isnull()
    ].groupby("job_location")["job_id"].count().sort_values(ascending=False)
)

Series([], Name: job_id, dtype: int64)

In [117]:
# Rename Column `job_location` to `regency_city`
internship_positions.rename(columns={"job_location": "regency_city"}, inplace=True)

In [118]:
# Fix some regency and city names
internship_positions["regency_city"] = (
    internship_positions.regency_city
    .str.replace(r"^Kab\s", r"Kab. ", regex=True)
    .str.replace("Pahuwato", "Pohuwato")
    .str.replace(r"^Kepulauan\s", r"Kab. Kep. ", regex=True)
    .str.replace(r"\sKepulauan\s", r" Kep. ", regex=True)
)

# 2. Data Conversion
Casting `weekly_working_day` to a categorical datatype due to its low cardinality (2 unique values).

In [119]:
# Cast the data type of `weekly_working_day` to string
internship_positions = internship_positions.astype({"weekly_working_day": "str"})

internship_positions.info()

<class 'pandas.DataFrame'>
RangeIndex: 28322 entries, 0 to 28321
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   job_id              28322 non-null  str  
 1   published_at        28322 non-null  str  
 2   job_title           28322 non-null  str  
 3   company             28322 non-null  str  
 4   regency_city        28322 non-null  str  
 5   education_level     28322 non-null  str  
 6   allowed_major       28322 non-null  str  
 7   job_description     28322 non-null  str  
 8   weekly_working_day  28322 non-null  str  
 9   requested_quota     28322 non-null  int64
 10  approved_quota      28322 non-null  int64
 11  applicant_count     28322 non-null  int64
 12  province            28322 non-null  str  
dtypes: int64(3), str(10)
memory usage: 20.0 MB


# 3. Feature Engineering

## 3.1 Feature Construction
Constructing the `acceptance_percentage` column (`100 * approved_quota / (1 + applicant_count)`)

In [120]:
# Create Column `acceptance_percentage`
internship_positions["acceptance_percentage"] = round(
    100
    * internship_positions["approved_quota"]
    / (internship_positions["applicant_count"].add(1)),
    2,
)

display(internship_positions.sample(10))

,job_id,published_at,job_title,company,regency_city,education_level,allowed_major,job_description,weekly_working_day,requested_quota,approved_quota,applicant_count,province,acceptance_percentage
8101,a22adf44-857d-4748-961b-d52c5ed61605,2026-07-16T09:57:34+07:00,Cyber Security Digital Governance,PT Bank Ina Perdana Tbk,Kota Adm. Jakarta Selatan,"Bachelor, Diploma","Teknik Informatika, Manajemen Keamanan dan Kes...",1. Membantu dalam penyusunan serta pemeliharaa...,5,2,2,10,DKI Jakarta,18.18
19163,a2419644-6d67-4c12-abf2-418a5dc872e4,2026-07-16T12:03:22+07:00,PENGELOLA KEHUMASAN,RUMAH TAHANAN NEGARA KELAS IIB HUMBANG HASUNDUTAN,Kab. Humbang Hasundutan,Bachelor,Komunikasi,1. Menyusun materi layanan informasi untuk med...,6,1,1,10,Sumatera Utara,9.09
3249,a226a3d9-5cc6-4404-9e53-3f2bcd5e5a5c,2026-07-16T10:16:01+07:00,KEPERAWATAN,PT BINTANG AMIN HUSADA (RS BINTANG AMIN),Kota Bandar Lampung,"Diploma, Bachelor, Profession","Keperawatan, Ilmu Keperawatan",Keperawatan merupakan unit yang bertanggungjaw...,5,30,30,109,Lampung,27.27
13917,a2312041-035f-401a-b85a-bfb157f92eb7,2026-07-16T10:07:11+07:00,Admin Project Management,Sigap Prima Astrea,Kota Adm. Jakarta Pusat,"Diploma, Bachelor","Manajemen, Teknik Industri, Administrasi",- Membantu administrasi seluruh project implem...,5,1,1,7,DKI Jakarta,12.50
10728,a231fc32-139d-40cd-a270-d21f905e829b,2026-07-16T10:24:39+07:00,SEO,PT. Commeasure Solutions Indonesia,Kota Adm. Jakarta Selatan,Bachelor,"Teknologi Informasi, Ilmu Komunikasi, Manajeme...",- Blog Content Creation & Optimization\n- Meny...,5,1,1,6,DKI Jakarta,14.29
16996,a243b870-49e4-4a4e-82b0-0b3c8841741e,2026-07-16T10:20:02+07:00,Service Management,PT United Tractors Tbk,Kota Adm. Jakarta Timur,"Diploma, Bachelor","Teknik Metalurgi, Teknik Mesin, Teknik Industr...",Membantu monitoring performa layanan kepada pe...,5,3,3,26,DKI Jakarta,11.11
17669,a243586d-5203-42cc-9f80-ac9a4404c893,2026-07-16T11:14:20+07:00,Hiring & Organization Development,PT. Rajawali Citra Televisi Indonesia,Kota Adm. Jakarta Barat,Bachelor,"Manajemen, Psikologi",1. Melaksanakan rangkaian proses seleksi dan r...,5,1,1,9,DKI Jakarta,10.00
15011,a232fecf-d7cb-4551-8216-a9d261adbf79,2026-07-16T11:06:31+07:00,HSSE,Universitas Padjadjaran (PTnbh),Kab. Sumedang,"Bachelor, Diploma, Profession","Kesehatan dan Keselamatan Kerja, Kesehatan Mas...",$24,5,3,3,23,Jawa Barat,12.50
4028,a2411099-1dc8-456b-98da-9a17cb723aa4,2026-07-16T11:13:13+07:00,Process Engineering Intern,Vktr Sakti Industries,Kab. Magelang,Bachelor,"Teknik Mesin, Teknik Industri, Teknik Manufaktur",Process Engineering Intern bertugas untuk:\n1....,5,1,1,4,Jawa Tengah,20.00
24735,a2436f1d-b3fb-4db5-8813-a2c9265f1ac5,2026-07-16T12:17:06+07:00,Asisten Pranata Komputer,BPS Kabupaten Asahan,Kab. Asahan,"Diploma, Bachelor","Teknik Informatika, Teknologi Informasi, Ilmu ...",Membantu pengelolaan sistem teknologi informas...,5,1,1,16,Sumatera Utara,5.88


## 3.2 Feature Transformation
### 3.2.1 Text Classification
* Categorizing `job_title` into a new `job_category` feature to reduce cardinality.

In [121]:
# Create a custom function to categorize the jobs
def categorize_job(title):
    if pd.isna(title):
        return "Uncategorized"
    
    t = str(title).lower()
    
    # 1. Healthcare & Medical (Added severe typos, hospital codes, and specialized terms)
    if any(w in t for w in ["perawat", "ners", "nurse", "psikiat", "spikiat", "psikat", "pskiat", "psikolog", "piskolog", "pskolog", "medis", "medic", "medik", "gizi", "nutri", "diet", "dokter", "doker", "doktor", "bidan", "apotek", "aptoker", "farmasi", "pharmac", "fisio", "physio", "radio", "sanitari", "sanitasi", "kesehatan", "promkes", "okupasi", "elektromedis", "atem", "terapi", "therap", "klinik", "atlm", "epidemiolog", "anestesi", "anastesi", "rekam medi", "perekam", "mr ", "casemix", "coder", "koder", "cssd", "ipsrs", "audiolog", "orthotic", "mcu", "ranap", "igd", "poliklinik", "vk ", "bersalin", "hemodialisa", "kardiovaskuler", "cardiovascular", "refraksi", "kebidanan", "keperawatan", "patologi", "mikrobiologi", "imunologi", "darah", "ambul", "hospital", "rehabilitasi", "admission"]):
        return "Healthcare & Medical"
    
    # 2. IT & Data
    elif any(w in t for w in ["komputer", "it ", " it", "programmer", "developer", "software", "data", "sistem", "system", "ui/", "/ux", "network", "cyber", "aplikasi", "application", "backend", "frontend", "website", "web", "informatika", "pusdatin", "jaringan", "ai ", "machine learning", "bda", "digital", "erp", "sap ", "helpdesk", "support it", "noc ", "rpa ", "command center", "dashboard", "cloud", "iot", "analytic"]):
        return "IT & Data"
        
    # 3. Engineering & Maintenance
    elif any(w in t for w in ["teknis", "technician", "maintenance", "engineer", "mekanik", "mechanic", "drafter", "drawing", "listrik", "sipil", "civil", "bangunan", "hvac", "otomotif", "mesin", "machine", "welder", "welding", "proyek", "project", "elektro", "electric", "arsitek", "architect", "maint", "equipment", "facility", "sarana", "prasarana", "geologi", "tambang", "mining", "instrument", "surveyor", "craft", "plumbing", "geofisika", "seismik", "geodesi", "geomatika", "toolman", "inspector", "inspektur", "konstruksi", "construction", "pipa", "baja", "otomasi", "automation"]):
        return "Engineering & Maintenance"
        
    # 4. Manufacturing, QA & Production
    elif any(w in t for w in ["produksi", "production", "operator", "qc", "qa", "quality", "pabrik", "manufacturing", "assembly", "packaging", "mutu", "plant", "molding", "mould", "mold", "pattern maker", "slitting", "blown film", "sewing", "garment", "textile", "printing", "finishing", "curing", "mixing", "extruder", "ppic", "rnd", "r&d", "research", "reserch", "set up", "cleanning", "rewinding", "laminasi", "improvement", "pdca", "koe ", "lean", "mill", "helper", "pe ", "ie "]):
        return "Manufacturing, QA & Production"
        
    # 5. Finance & Banking
    elif any(w in t for w in ["keuangan", "akuntansi", "accounting", "akuntan", "pajak", "tax", "auditor", "audit", "bendahara", "anggaran", "finance", "treasury", "billing", "kasir", "cashier", "credit", "kredit", "loan", "pembiayaan", "funding", "transaction", "collection", "receivable", "payable", "wealth", "insurance", "asuransi", "actuary", "aktuaria", "bank", "bni", "teller", "pawning", "micro", "invest", "budget", "cost "]):
        return "Finance & Banking"
        
    # 6. Sales, Marketing & Hospitality
    elif any(w in t for w in ["barista", "cook", "koki", "pastry", "bakery", "culinary", "chef", "layanan", "frontliner", "frontlner", "sales", "marketing", "f&b", "fb ", "store", "customer", "receptionist", "pemasaran", "pramusaji", "hotel", "event", "reservation", "guest", "hospitality", "dancer", "entertainment", "commercial", "merchandis", "retail", "promot", "promosi", "brand", "business development", "bd ", "partnership", "account executive", "masak", "food", "beverage", "catering", "kitchen", "tour ", "travel", "room", "bro", "activation"]):
        return "Sales, Marketing & Hospitality"
        
    # 7. Media, PR & Creative
    elif any(w in t for w in ["humas", "kehumasan", "design", "desain", "kreatif", "creative", "video", "animator", "animation", "content", "konten", "sosial media", "social media", "sosmed", "kol ", "publisitas", "jurnalis", "journalist", "wartawan", "editor", "multimedia", "reporter", "fotografer", "photographer", "broadcasting", "komunikasi", "communication", "visual", "vm artist", "motion", "copywriter", "writer", "art ", "talent", "audio", "camera", "campaign", "publikasi", "publik", "illustrator", "media", "broadcast", "creator"]):
        return "Media, PR & Creative"
        
    # 8. Legal, Risk & Compliance
    elif any(w in t for w in ["hukum", "legal", "law", "compliance", "kepatuhan", "risk", "risiko", "hse", "hsse", "qhse", "she", "ehs", "safety", "k3", "security", "keamanan", "fraud", "investigasi", "pengaduan", "maladministrasi", "litigasi", "regulas", "regulatory", "sertifikasi", "perizinan", "izin", "kekayaan intelektual"]):
        return "Legal, Risk & Compliance"
        
    # 9. Logistics & Supply Chain
    elif any(w in t for w in ["gudang", "warehouse", "logistik", "logistic", "exim", "supply chain", "scm", "inventory", "pengadaan", "purchasing", "procurement", "buyer", "ekspor", "impor", "export", "import", "cargo", "shipping", "freight", "delivery", "transport", "fleet", "ekspeditor", "harbour", "port ", "bandara", "airport", "pelabuhan", "aviation", "aero", "aircraft", "checker", "terminal"]):
        return "Logistics & Supply Chain"
        
    # 10. Education, Training & Government
    elif any(w in t for w in ["kebijakan", "pemerintahan", "penelaah", "pengawas", "asn", "biro", "kementerian", "pemda", "pns", "diplomat", "instruktur", "pelatihan", "pembelajaran", "tentor", "pengajar", "edukator", "diklat", "akademik", "guru", "dosen", "widyaiswara", "pusat", "badan", "tutor", "statistik", "statistisi", "peneliti", "pustaka", "kearsipan", "arsip", "kurator", "laporan", "pelaporan", "penyusun", "evaluasi", "pengolah", "dokumen", "evaluator", "pemeriksaan"]):
        return "Education, Training & Government"
        
    # 11. Agriculture & Environment
    elif any(w in t for w in ["pertanian", "perikanan", "peternakan", "perkebunan", "agribisnis", "kehutanan", "lingkungan", "agronomi", "pangan", "tambak", "tanaman", "kebun", "hewan", "hutan", "forestry", "environment", "sustainability", "esg", "limbah", "waste", "marine", "hydro", "iklim", "climate", "budidaya", "ternak", "nelayan", "satwa", "flora", "fauna", "ekologi", "air "]):
        return "Agriculture & Environment"
        
    # 12. Language & Translation
    elif any(w in t for w in ["isyarat", "penerjemah", "translator", "interpreter", "language", "mandarin", "japanese", "english", "bahasa"]):
        return "Language & Translation"
        
    # 13. Correctional & Social Services
    elif any(w in t for w in ["pembinaan", "kepribadian", "pembimbing kemasyarakatan", "warga binaan", "kegiatan kerja", "rohani", "sosial", "pemasyarakatan", "community", "csr", "tjsl", "bina", "klien", "konselor"]):
        return "Correctional & Social Services"

    # 14. HR, Admin & Management (General catch-alls placed at the very end)
    elif any(w in t for w in ["sdm", "human resource", "hr", "ga", "general affair", "administrasi", "admin", "bmn", "sekretaris", "secretary", "tata usaha", "tu ", "personil", "personalia", "rekrutmen", "recruitment", "talent acquisition", "od ", "organization development", "umum", "fasilitas", "manajemen", "management", "manager", "pmo", "strategi", "koordinator", "coordinator", "supervisor", "spv", "director", "operasional", "operation", "asset", "aset", "clerical", "sarana", "pejabat", "pengelola", "asisten", "officer", "staff", "staf", "pelaksana", "magang", "intern", "consultant", "konsultan", "planner"]):
        return "HR, Admin & Management"
        
    else:
        return "Other"

In [122]:
# Apply the function to create the new column
internship_positions["job_category"] = internship_positions["job_title"].apply(categorize_job)

# Verify the distribution of the new categories
display(internship_positions["job_category"].value_counts())

job_category
HR, Admin & Management              5176
Healthcare & Medical                4063
IT & Data                           2802
Media, PR & Creative                2570
Sales, Marketing & Hospitality      2150
Correctional & Social Services      2005
Finance & Banking                   1707
Engineering & Maintenance           1701
Education, Training & Government    1294
Manufacturing, QA & Production      1083
Legal, Risk & Compliance            1036
Other                                994
Logistics & Supply Chain             837
Agriculture & Environment            597
Language & Translation               307
Name: count, dtype: int64

* Engineering 9 new boolean flag columns (e.g., `allows_it_and_computer_majors`) by grouping over 1,300 distinct majors in the `allowed_major` column using regex pattern matching.

In [123]:
# Create new Boolean columns
# Define the categories and their specific Indonesian Regex keywords
maj_categories = {
    "it_and_computer": r"informatika|komputer|sistem informasi|perangkat lunak|multimedia|jaringan|siber|data|teknologi informasi|piranti lunak|website",
    "engineering": r"teknik(?!\s*(?:informatika|komputer|multimedia))|rekayasa(?!\s*(?:perangkat lunak|internet|komputer))|arsitektur|mesin|elektro|sipil|industri|mekatronika|otomotif|manufaktur|konstruksi|geodesi|geologi|tambang|perkapalan|dirgantara|nautika|listrik|kelistrikan|logam|tekstil|metrologi|instrumentasi|perencanaan|planologi|tata ruang",
    "business": r"manajemen|akuntansi|bisnis|ekonomi|keuangan|administrasi|adminsitrasi|logistik|pemasaran|marketing|pajak|perbankan|retail|niaga|aktiva|kewirausahaan|asuransi",
    "health": r"kedokteran|keperawatan|kebidanan|farmasi|kesehatan|gizi|medik|medis|terapi|radiologi|klinik|apoteker|sanitasi|higiene|hiperkes|optisi|optometri|ortotik|prostetik|darah|audiologi|akupunktur|herbal|rumah sakit|nutrisi",
    "science_and_math": r"matematika|statistik|statistika|biologi|kimia|fisika|sains|aktuaria|geografi|astronomi|lingkungan|bumi|kartografi|penginderaan|oseanografi",
    "agriculture_and_fisheries": r"agribisnis|agribinis|pertanian|peternakan|perikanan|kehutanan|agroteknologi|agroekoteknologi|agro|perkebunan|agronomi|hortikultura|hewan|laut|budidaya|tanaman|pangan|pertanahan",
    "arts_and_media": r"desain|seni|komunikasi|film|televisi|jurnalistik|penyiaran|broadcasting|hubungan masyarakat|humas|fotografi|kriya|tari|musik|karawitan|animasi|media|audio|video|penerbitan",
    "social_and_law": r"hukum|sosiologi|psikologi|sastra|bahasa|kriminologi|kesejahteraan|pemerintahan|politik|hubungan internasional|sejarah|filsafat|antropologi|perpustakaan|kearsipan|arsip|agama|teologi|syariah|islam|kristen|buddha|hindu",
    "education": r"pendidikan|pgsd|pgpaud|tadris|bimbingan|konseling|tarbiyah|guru|kependidikan|penyuluhan",
    "tourism_and_hospitality": r"pariwisata|perhotelan|tata boga|tata rias|tata busana|fashion|kuliner|wisata|mice|travel|hospitaliti|hidang|patiseri"
}

# Ensure the column is treated as a string and handle missing values
internship_positions["allowed_major"] = internship_positions["allowed_major"].fillna("")

# Iterate through the dictionary to create the new Boolean columns
for cat, pattern in maj_categories.items():
    col_name = f"allows_{cat}_majors"
    
    # Check if any keyword in the pattern exists in the "allowed_major" string
    mask = internship_positions["allowed_major"].str.contains(pattern, case=False, regex=True)    
    internship_positions[col_name] = np.where(mask, "Yes", "No")

# Preview the results
display(internship_positions.sample(10))

,job_id,published_at,job_title,company,regency_city,education_level,allowed_major,job_description,weekly_working_day,requested_quota,...,allows_it_and_computer_majors,allows_engineering_majors,allows_business_majors,allows_health_majors,allows_science_and_math_majors,allows_agriculture_and_fisheries_majors,allows_arts_and_media_majors,allows_social_and_law_majors,allows_education_majors,allows_tourism_and_hospitality_majors
28114,a2436eac-e1f0-446c-a903-76ea827fc9dc,2026-07-16T10:29:37+07:00,Internship - bag. Rendal & Man Proyek,Pupuk Sriwidjaja Palembang,Kota Palembang,"Bachelor, Diploma","Teknik Informatika, Teknik Industri, Teknik Sipil","• Memahami dalam bidang Manajemen Proyek, pros...",5,1,...,Yes,Yes,No,No,No,No,No,No,No,No
23593,a239313f-e217-4680-968b-708c621a4a5b,2026-07-16T10:31:05+07:00,Admin Logistik,Baba Rafi Internasional,Kab. Sidoarjo,"Diploma, Bachelor","Manajemen, Teknik Industri, Administrasi Bisni...",Program pelatihan yang dirancang untuk membeka...,5,1,...,Yes,Yes,Yes,No,No,No,No,No,No,No
4511,a2335fc7-e7f5-41a8-8886-372a8b4a0411,2026-07-16T09:44:06+07:00,Sosmed (Video Editor),PT. Arkadia Media Nusantara,Kota Adm. Jakarta Barat,"Diploma, Bachelor","Multimedia, Penyuntingan Audio dan Video, Desa...",• Pendidikan minimal D3/S1 semua jurusan.\n• M...,6,1,...,Yes,No,No,No,No,No,Yes,No,No,No
22414,a23dbeaf-739f-4129-a0ba-68d0650700b3,2026-07-16T13:17:14+07:00,Pelaksana Optimalisasi Pemanfaatan Teknologi I...,Balai Standardisasai dan Pelayanan Jasa Indust...,Kota Bandar Lampung,Bachelor,Teknologi Hasil Pertanian,$26,5,1,...,No,No,No,No,No,Yes,No,No,No,No
16443,a2420ded-232e-4dc5-9e07-a399c5ac81c2,2026-07-16T12:35:08+07:00,Staf Bidang Promosi dan Kerja Sama,"Balai Pelatihan Sumber Daya Manusia Metrologi,...",Kab. Bandung Barat,Bachelor,"Administrasi BIsnis, Hubungan Masyarakat, Ilmu...",Mendukung pelaksanaan pengumpulan data dan adm...,5,2,...,No,No,Yes,No,No,No,Yes,No,No,No
16698,a224e9d0-3aa0-4c2c-9e12-b155fdd47464,2026-07-16T10:24:43+07:00,REPORTER CETAK DAN ONLINE,PT. Wahana Semesta Lampung,Kota Bandar Lampung,Bachelor,"Ilmu Komunikasi, Manajemen Ekonomi, Ilmu Hukum","Proyeksi berita, liputan berita sesuai proyeks...",6,5,...,No,No,Yes,No,No,No,Yes,Yes,No,No
14536,a243a83d-5b57-44fb-b86f-d9395d6f6488,2026-07-16T11:25:31+07:00,Information and Communication Technology,"PT Semen Baturaja (Persero) Tbk, Pabrik Baturaja",Kab. Ogan Komering Ulu,Bachelor,"Teknik Informatika, Pendidikan Sistem dan Tekn...",1.Memahami fungsi dan peran ICT dalam mendukun...,5,4,...,Yes,No,No,No,No,No,No,No,Yes,No
11166,a22ad2cc-fa06-4e19-a5d4-38c88b7dfdee,2026-07-16T09:50:46+07:00,Content Marketing,PT Indonusa Telemedia,Kota Adm. Jakarta Selatan,"Diploma, Bachelor","Jurnalistik, Ilmu Komunikasi, Komunikasi Digit...","a.\tMelakukan pengumpulan, penelusuran, dan an...",5,1,...,No,No,Yes,No,No,No,Yes,No,No,No
12282,a240df97-4f8b-4c25-a11f-8f513547bf8e,2026-07-16T09:47:30+07:00,Warehouse & Logistic Management Staff,Sagara Prima Perkasa,Kota Tangerang Selatan,"Diploma, Bachelor","Manajemen, Teknik Industri, Logistik Bisnis, M...",Peserta magang akan mempelajari dan membantu p...,5,2,...,No,Yes,Yes,No,Yes,No,No,No,No,No
10760,a2419f91-4e29-42a8-a60d-5e233e6ef0ff,2026-07-16T10:22:02+07:00,Bagian Humas & Branding Korporasi,Asuransi Asei Indonesia,Kota Adm. Jakarta Selatan,Bachelor,"Ilmu Komunikasi, Desain Komunikasi Visual, Hub...",Tanggung Jawab Utama: \n- Membina hubungan bai...,5,1,...,No,No,No,No,No,No,Yes,No,No,No


In [124]:
# Review the rows that slipped through the Regex patterns
category_cols = [f"allows_{cat}_majors" for cat in maj_categories.keys()]
uncategorized_mask = (internship_positions[category_cols] == "No").all(axis=1)

uncategorized_positions = internship_positions[uncategorized_mask]

print(uncategorized_positions['allowed_major'].unique())

<ArrowStringArray>
[]
Length: 0, dtype: str


### 3.2.2 Binning (Discretization)
Binning all heavy right-skewed numericals (`requested_quota`, `approved_quota`, `applicant_count`, `acceptance_percentage`) to capture "whale" postings in an "Extreme" category without deleting them.

In [125]:
# Bin `requested_quota` and `approved_quota`
quota_edges = [1, 2, 10, 50, np.inf]
quota_labels = ["1 to 2", "3 to 10", "11 to 50", "50+"]

internship_positions["requested_quota_category"] = pd.cut(
    internship_positions["requested_quota"],
    bins=quota_edges,
    labels=quota_labels,
    include_lowest=True
)

internship_positions["approved_quota_category"] = pd.cut(
    internship_positions["approved_quota"],
    bins=quota_edges,
    labels=quota_labels,
    include_lowest=True
)

display(internship_positions.head())

,job_id,published_at,job_title,company,regency_city,education_level,allowed_major,job_description,weekly_working_day,requested_quota,...,allows_business_majors,allows_health_majors,allows_science_and_math_majors,allows_agriculture_and_fisheries_majors,allows_arts_and_media_majors,allows_social_and_law_majors,allows_education_majors,allows_tourism_and_hospitality_majors,requested_quota_category,approved_quota_category
0,a240f2ba-12c0-4958-b416-c3e9c1d4e344,2026-07-16T12:58:55+07:00,PSIKOLOG,RUMAH TAHANAN NEGARA KELAS IIB SIBUHUAN,Kab. Padang Lawas,Profession,Psikologi,1. Melakukan asesmen psikologis terhadap anak ...,6,1,...,No,No,No,No,No,Yes,No,No,1 to 2,1 to 2
1,a240f336-b6b7-47fe-8096-5b4c2eb2ed1f,2026-07-16T12:58:55+07:00,PSIKIATER,RUMAH TAHANAN NEGARA KELAS IIB SIBUHUAN,Kab. Padang Lawas,Profession,Kedokteran,1. Menangani gangguan kesehatan jiwa warga bin...,6,1,...,No,Yes,No,No,No,No,No,No,1 to 2,1 to 2
2,a242e6a4-2c7a-4ce4-8a57-7aff601a5e2c,2026-07-16T12:57:40+07:00,PSIKIATER,RUMAH TAHANAN NEGARA KELAS IIB SALATIGA,Kota Salatiga,Profession,Kedokteran,1. Menangani gangguan kesehatan jiwa warga bin...,5,1,...,No,Yes,No,No,No,No,No,No,1 to 2,1 to 2
3,a24138aa-bcf6-4940-bfaf-dfe5dbf66ca6,2026-07-16T12:50:42+07:00,PERAWAT KESEHATAN,LEMBAGA PEMASYARAKATAN KELAS III SUKAMARA,Kab. Sukamara,Bachelor,Ilmu Gizi,1. Memberikan perawatan kesehatan umum dan tin...,6,1,...,No,Yes,No,No,No,No,No,No,1 to 2,1 to 2
4,a23f7c52-9123-46e9-bb5e-be376b1d77f2,2026-07-16T12:38:52+07:00,Psikiater,LEMBAGA PEMASYARAKATAN KELAS III ARJASA,Kab. Sumenep,Bachelor,Kedokteran,1. Menangani gangguan kesehatan jiwa warga bin...,6,1,...,No,Yes,No,No,No,No,No,No,1 to 2,1 to 2


In [126]:
# Bin Column `applicant_count`
applicant_edges = [0, 5, 10, 20, 50, np.inf]
applicant_labels = ["0 to 5", "6 to 10", "11 to 20", "21 to 50", "50+"]

internship_positions["applicant_count_category"] = pd.cut(
    internship_positions["applicant_count"],
    bins=applicant_edges,
    labels=applicant_labels,
    include_lowest=True
)

display(internship_positions.head())

,job_id,published_at,job_title,company,regency_city,education_level,allowed_major,job_description,weekly_working_day,requested_quota,...,allows_health_majors,allows_science_and_math_majors,allows_agriculture_and_fisheries_majors,allows_arts_and_media_majors,allows_social_and_law_majors,allows_education_majors,allows_tourism_and_hospitality_majors,requested_quota_category,approved_quota_category,applicant_count_category
0,a240f2ba-12c0-4958-b416-c3e9c1d4e344,2026-07-16T12:58:55+07:00,PSIKOLOG,RUMAH TAHANAN NEGARA KELAS IIB SIBUHUAN,Kab. Padang Lawas,Profession,Psikologi,1. Melakukan asesmen psikologis terhadap anak ...,6,1,...,No,No,No,No,Yes,No,No,1 to 2,1 to 2,0 to 5
1,a240f336-b6b7-47fe-8096-5b4c2eb2ed1f,2026-07-16T12:58:55+07:00,PSIKIATER,RUMAH TAHANAN NEGARA KELAS IIB SIBUHUAN,Kab. Padang Lawas,Profession,Kedokteran,1. Menangani gangguan kesehatan jiwa warga bin...,6,1,...,Yes,No,No,No,No,No,No,1 to 2,1 to 2,0 to 5
2,a242e6a4-2c7a-4ce4-8a57-7aff601a5e2c,2026-07-16T12:57:40+07:00,PSIKIATER,RUMAH TAHANAN NEGARA KELAS IIB SALATIGA,Kota Salatiga,Profession,Kedokteran,1. Menangani gangguan kesehatan jiwa warga bin...,5,1,...,Yes,No,No,No,No,No,No,1 to 2,1 to 2,0 to 5
3,a24138aa-bcf6-4940-bfaf-dfe5dbf66ca6,2026-07-16T12:50:42+07:00,PERAWAT KESEHATAN,LEMBAGA PEMASYARAKATAN KELAS III SUKAMARA,Kab. Sukamara,Bachelor,Ilmu Gizi,1. Memberikan perawatan kesehatan umum dan tin...,6,1,...,Yes,No,No,No,No,No,No,1 to 2,1 to 2,0 to 5
4,a23f7c52-9123-46e9-bb5e-be376b1d77f2,2026-07-16T12:38:52+07:00,Psikiater,LEMBAGA PEMASYARAKATAN KELAS III ARJASA,Kab. Sumenep,Bachelor,Kedokteran,1. Menangani gangguan kesehatan jiwa warga bin...,6,1,...,Yes,No,No,No,No,No,No,1 to 2,1 to 2,0 to 5


In [127]:
# Bin Column `acceptance_percentage`
acceptance_edges = [0, 10, 25, 50, np.inf]
acceptance_labels = ["0 - 10%", "11 - 25%", "26 - 50%", "50%+"]

internship_positions["acceptance_percentage_category"] = pd.cut(
    internship_positions["acceptance_percentage"],
    bins=acceptance_edges,
    labels=acceptance_labels,
    include_lowest=True
)

display(internship_positions.head())

,job_id,published_at,job_title,company,regency_city,education_level,allowed_major,job_description,weekly_working_day,requested_quota,...,allows_science_and_math_majors,allows_agriculture_and_fisheries_majors,allows_arts_and_media_majors,allows_social_and_law_majors,allows_education_majors,allows_tourism_and_hospitality_majors,requested_quota_category,approved_quota_category,applicant_count_category,acceptance_percentage_category
0,a240f2ba-12c0-4958-b416-c3e9c1d4e344,2026-07-16T12:58:55+07:00,PSIKOLOG,RUMAH TAHANAN NEGARA KELAS IIB SIBUHUAN,Kab. Padang Lawas,Profession,Psikologi,1. Melakukan asesmen psikologis terhadap anak ...,6,1,...,No,No,No,Yes,No,No,1 to 2,1 to 2,0 to 5,50%+
1,a240f336-b6b7-47fe-8096-5b4c2eb2ed1f,2026-07-16T12:58:55+07:00,PSIKIATER,RUMAH TAHANAN NEGARA KELAS IIB SIBUHUAN,Kab. Padang Lawas,Profession,Kedokteran,1. Menangani gangguan kesehatan jiwa warga bin...,6,1,...,No,No,No,No,No,No,1 to 2,1 to 2,0 to 5,50%+
2,a242e6a4-2c7a-4ce4-8a57-7aff601a5e2c,2026-07-16T12:57:40+07:00,PSIKIATER,RUMAH TAHANAN NEGARA KELAS IIB SALATIGA,Kota Salatiga,Profession,Kedokteran,1. Menangani gangguan kesehatan jiwa warga bin...,5,1,...,No,No,No,No,No,No,1 to 2,1 to 2,0 to 5,50%+
3,a24138aa-bcf6-4940-bfaf-dfe5dbf66ca6,2026-07-16T12:50:42+07:00,PERAWAT KESEHATAN,LEMBAGA PEMASYARAKATAN KELAS III SUKAMARA,Kab. Sukamara,Bachelor,Ilmu Gizi,1. Memberikan perawatan kesehatan umum dan tin...,6,1,...,No,No,No,No,No,No,1 to 2,1 to 2,0 to 5,50%+
4,a23f7c52-9123-46e9-bb5e-be376b1d77f2,2026-07-16T12:38:52+07:00,Psikiater,LEMBAGA PEMASYARAKATAN KELAS III ARJASA,Kab. Sumenep,Bachelor,Kedokteran,1. Menangani gangguan kesehatan jiwa warga bin...,6,1,...,No,No,No,No,No,No,1 to 2,1 to 2,0 to 5,50%+


## 3.3 Feature Encoding
One-hot encoding `education_level` for downstream stakeholder consumption.

In [128]:
# One hot encode `education_level`
ed_level_dummies = internship_positions["education_level"].str.lower().str.get_dummies(sep=", ")
ed_level_dummies = ed_level_dummies.replace({0: "No", 1: "Yes"}).add_prefix("allows_")
ed_level_dummies = ed_level_dummies.add_suffix("_level")

internship_positions = pd.concat([internship_positions, ed_level_dummies], axis=1)
display(internship_positions.sample(5))

,job_id,published_at,job_title,company,regency_city,education_level,allowed_major,job_description,weekly_working_day,requested_quota,...,allows_social_and_law_majors,allows_education_majors,allows_tourism_and_hospitality_majors,requested_quota_category,approved_quota_category,applicant_count_category,acceptance_percentage_category,allows_bachelor_level,allows_diploma_level,allows_profession_level
21810,a229442e-98bd-470d-93dc-2647f6fa4d88,2026-07-16T11:16:48+07:00,Business Development,Dynamic Talenta Navigator,Kota Semarang,"Bachelor, Diploma","Bisnis, Manajemen Bisnis, Manajemen Pemasaran/...",Business Development Intern bertanggung jawab ...,5,1,...,No,No,No,1 to 2,1 to 2,11 to 20,0 - 10%,Yes,Yes,No
8081,a24370c6-b787-49be-a0ee-c5cf8383a5e6,2026-07-16T10:04:04+07:00,Workshop (Electrical Automation),Akebono Brake Astra Indonesia,Kota Adm. Jakarta Utara,"Bachelor, Diploma","Teknik Mekatronika, Teknik Mesin, Teknik Instr...",-Mempelajari administrasi Workshop\n-Melakukan...,5,2,...,No,No,No,1 to 2,1 to 2,6 to 10,11 - 25%,Yes,Yes,No
12087,a2418465-3f0d-4655-9aca-6efca9b6000c,2026-07-16T12:18:43+07:00,Asisten Statistisi,BPS Kabupaten Bantaeng,Kab. Bantaeng,"Diploma, Bachelor, Profession","Sains Data, Aktuaria, Statistika, Matematika","Membantu pengumpulan, pengolahan, verifikasi, ...",5,2,...,No,No,No,1 to 2,1 to 2,11 to 20,11 - 25%,Yes,Yes,Yes
15823,a241a953-e61b-428a-82e8-18c0c0f347a3,2026-07-16T11:00:54+07:00,Performance and Investment Management,PT Perusahaan Gas Negara Tbk,Kota Adm. Jakarta Barat,Bachelor,"Manajemen Keuangan, Manajemen, Teknik Industri...",Mendukung monitoring kinerja perusahaan dan ev...,5,1,...,No,No,No,1 to 2,1 to 2,6 to 10,11 - 25%,Yes,No,No
23981,a24112b2-fefb-4e8b-99eb-9655d1fe5147,2026-07-16T13:12:52+07:00,Administrasi Klaim,BPJS Kesehatan Kantor Cabang Jakarta Pusat,Kota Adm. Jakarta Pusat,"Diploma, Bachelor","Keperawatan, Administrasi Rumah Sakit, Farmasi...",Membantu pelaksanaan kegiatan administratif da...,5,1,...,No,No,No,1 to 2,1 to 2,11 to 20,0 - 10%,Yes,Yes,No


## 3.4 Feature Extraction
Extracting a new binary flag, `allows_all_majors`, by parsing the `job_description` column.

In [129]:
all_majors_condition = internship_positions.job_description.str.contains(
    r"semua\sjurusan|jurusan\sapa.*|all\smajors|any\smajor",
    case=False
)

internship_positions["allows_all_majors"] = np.where(all_majors_condition, "Yes", "No")

display(internship_positions.sample(5))

,job_id,published_at,job_title,company,regency_city,education_level,allowed_major,job_description,weekly_working_day,requested_quota,...,allows_education_majors,allows_tourism_and_hospitality_majors,requested_quota_category,approved_quota_category,applicant_count_category,acceptance_percentage_category,allows_bachelor_level,allows_diploma_level,allows_profession_level,allows_all_majors
24000,a2413d14-f245-4e88-b121-dc70d37a0a2f,2026-07-16T12:58:18+07:00,PENGELOLA BMN,BALAI PEMASYARAKATAN KELAS I SURABAYA,Kab. Sidoarjo,Bachelor,"Manajemen Aset Publik, Kepelatihan Olahraga, S...",1. Mengelola aset dan inventaris milik negara ...,5,1,...,No,No,1 to 2,1 to 2,11 to 20,0 - 10%,Yes,No,No,No
6442,a2437f9f-6acd-4bb0-bba3-7ed489576923,2026-07-16T12:03:00+07:00,Staf Penelaah Teknis\tpada Asdep Pengembangan ...,Kementerian Koordinator Bidang Perekonomian,Kota Adm. Jakarta Pusat,Bachelor,"Teknik Informatika, Desain Komunikasi Visual, ...",Melaksanakan dukungan teknis dalam rangka peny...,5,1,...,No,No,1 to 2,1 to 2,0 to 5,11 - 25%,Yes,No,No,No
6940,a241d474-316f-42aa-a67a-7c712196025c,2026-07-16T10:40:19+07:00,Procurement Data & Digital Support Intern,PT. Graha Sarana Duta,Kota Adm. Jakarta Pusat,"Diploma, Bachelor, Profession","Teknik Informatika, Ilmu Komputer, Sistem Info...",Peserta magang akan mendukung administrasi dan...,5,1,...,No,No,1 to 2,1 to 2,0 to 5,11 - 25%,Yes,Yes,Yes,No
18546,a23f1712-6508-4853-a702-7bb16913bf83,2026-07-16T12:15:40+07:00,PEMBINA KEPRIBADIAN,LEMBAGA PEMASYARAKATAN KELAS IIB PENYABUNGAN,Kab. Mandailing Natal,Bachelor,"Seni Tari, Seni Musik",1.\tMenyusun dan melaksanakan program pembinaa...,6,2,...,No,No,1 to 2,1 to 2,11 to 20,0 - 10%,Yes,No,No,No
10659,a2373707-ffdd-44f5-aee9-2228ae5d1db8,2026-07-16T10:29:57+07:00,Administrasi,Koperasi Jasa Daya Cipta Sejati,Kota Depok,Diploma,"Akuntansi Perpajakan, Akuntansi Keuangan, Admi...",$25,6,1,...,No,No,1 to 2,1 to 2,6 to 10,11 - 25%,No,Yes,No,No


# 4. Schema Finalization
Reorganizing the final 14 columns into a logical analytical structure before exporting.

In [130]:
# Get all the columns
internship_positions.columns

Index(['job_id', 'published_at', 'job_title', 'company', 'regency_city',
       'education_level', 'allowed_major', 'job_description',
       'weekly_working_day', 'requested_quota', 'approved_quota',
       'applicant_count', 'province', 'acceptance_percentage', 'job_category',
       'allows_it_and_computer_majors', 'allows_engineering_majors',
       'allows_business_majors', 'allows_health_majors',
       'allows_science_and_math_majors',
       'allows_agriculture_and_fisheries_majors',
       'allows_arts_and_media_majors', 'allows_social_and_law_majors',
       'allows_education_majors', 'allows_tourism_and_hospitality_majors',
       'requested_quota_category', 'approved_quota_category',
       'applicant_count_category', 'acceptance_percentage_category',
       'allows_bachelor_level', 'allows_diploma_level',
       'allows_profession_level', 'allows_all_majors'],
      dtype='str')

In [131]:
# Reorganize the position of the columns
final_cols = [
    "job_id",
    "published_at",
    "job_title",
    "job_category",
    "company",
    "regency_city",
    "province",
    "allows_bachelor_level",
    "allows_diploma_level",
    "allows_profession_level",
    "allowed_major",
    "allows_it_and_computer_majors",
    "allows_engineering_majors",
    "allows_business_majors",
    "allows_health_majors",
    "allows_science_and_math_majors",
    "allows_agriculture_and_fisheries_majors",
    "allows_arts_and_media_majors",
    "allows_social_and_law_majors",
    "allows_education_majors",
    "allows_tourism_and_hospitality_majors",
    "allows_all_majors",
    "job_description",
    "weekly_working_day",
    "requested_quota_category",
    "approved_quota_category",
    "applicant_count_category",
    "acceptance_percentage_category",
    "requested_quota",
    "approved_quota",
    "applicant_count",
    "acceptance_percentage",
]

internship_postings = internship_positions[final_cols]

display(internship_postings.sample(10))

,job_id,published_at,job_title,job_category,company,regency_city,province,allows_bachelor_level,allows_diploma_level,allows_profession_level,...,job_description,weekly_working_day,requested_quota_category,approved_quota_category,applicant_count_category,acceptance_percentage_category,requested_quota,approved_quota,applicant_count,acceptance_percentage
7141,a23529e4-44bc-4e4f-ac21-4e8abc78c000,2026-07-16T10:24:27+07:00,Staff Accounting,Finance & Banking,PT. Wahana Prestasi Logistik,Kota Tangerang Selatan,Banten,Yes,No,No,...,- Membantu proses pencatatan transaksi keuanga...,6,1 to 2,1 to 2,0 to 5,11 - 25%,1,1,5,16.67
23372,a242ffe3-7310-4649-97d5-b8e8730c11d5,2026-07-16T12:33:05+07:00,ASISTEN PENGEMBANGAN WEB,IT & Data,KANIM KELAS I NON TPI PATI,Kab. Pati,Jawa Tengah,Yes,No,No,...,1. Menyusun dan mengelola prosedur kerja serta...,5,1 to 2,1 to 2,11 to 20,0 - 10%,1,1,14,6.67
4774,a23f6282-8f15-456c-86d7-da2b6f32b8ea,2026-07-16T10:19:02+07:00,Intern Maintenance,Engineering & Maintenance,Southeast Asia Pipe Industries,Kab. Lampung Selatan,Lampung,Yes,Yes,No,...,-Membantu tim Maintenance dalam dukungan tekni...,5,1 to 2,1 to 2,6 to 10,11 - 25%,2,2,8,22.22
17174,a2436da2-4f0b-4c2e-8ae1-9d247f44e753,2026-07-16T12:56:11+07:00,PENGELOLA SDM,"HR, Admin & Management",LEMBAGA PEMASYARAKATAN PEREMPUAN KELAS IIA JAK...,Kota Adm. Jakarta Timur,DKI Jakarta,Yes,No,No,...,1. Mengumpulkan data dan informasi yang releva...,5,1 to 2,1 to 2,6 to 10,0 - 10%,1,1,9,10.00
27578,a23f658f-cd26-4c89-becd-14bdc55cb08b,2026-07-16T12:36:05+07:00,Asisten Arsiparis,"Education, Training & Government",BPS Kabupaten Tanah Datar,Kab. Tanah Datar,Sumatera Barat,Yes,No,No,...,"Mendukung penyusunan, pengelolaan, penyimpanan...",5,1 to 2,1 to 2,21 to 50,0 - 10%,1,1,27,3.57
2982,a240ea32-691b-460f-ab46-cd405911bf85,2026-07-16T12:48:47+07:00,Pengelola Kegiatan Kerja,Correctional & Social Services,LEMBAGA PEMASYARAKATAN KELAS III RANGKASBITUNG,Kab. Lebak,Banten,Yes,No,No,...,$28,6,1 to 2,1 to 2,6 to 10,11 - 25%,2,2,7,25.00
11960,a241f2a4-1857-491f-ad2f-3114589b7f05,2026-07-16T20:15:50+07:00,Assistant Producer - IDX,Other,Mnc Televisi Network,Kota Adm. Jakarta Pusat,DKI Jakarta,Yes,No,No,...,$26,5,1 to 2,1 to 2,11 to 20,11 - 25%,2,2,13,14.29
3416,a241459a-f8ea-4374-876f-3b19209feb73,2026-07-16T16:24:19+07:00,Paper Finishing Apprenticeship,"Manufacturing, QA & Production",PT Bukit Muria Jaya,Kab. Karawang,Jawa Barat,Yes,No,No,...,Memahami Proses End-to-End Area Paper Finishin...,5,1 to 2,1 to 2,0 to 5,11 - 25%,1,1,4,20.00
9181,a2419589-ac83-414c-8bcf-e21986f569a2,2026-07-16T10:56:17+07:00,Staff Supply planning,"HR, Admin & Management",Cahaya Abadi Plastik,Kab. Bekasi,Jawa Barat,Yes,Yes,No,...,Merencanakan dan mengendalikan kebutuhan bahan...,5,3 to 10,3 to 10,21 to 50,11 - 25%,4,4,23,16.67
24759,a2415b69-1457-4432-b7b2-c1f5b9cf8741,2026-07-16T12:08:23+07:00,Staf Komunikasi Dan Kesekretariatan,"Media, PR & Creative",BPJS Kesehatan Kantor Cabang Solok,Kota Solok,Sumatera Barat,Yes,Yes,No,...,Membantu melaksanakan kegiatan administratif d...,5,1 to 2,1 to 2,11 to 20,0 - 10%,1,1,16,5.88


In [132]:
# Load the final, clean data to a local directory
internship_postings.to_parquet(
    INTERIM_DATA_DIR / "internship_postings.parquet", index=False
) 